# ECMM422J Coursework 2 — California Housing Price Prediction

**Student ID:** XXXXXXXX

This notebook develops unsupervised and supervised machine learning models to
predict the median house value of California districts (1990 U.S. Census data).

**Pipeline:**
1. Setup
2. Data loading
3. Exploratory data analysis
4. Preprocessing & feature engineering
5. Visualisation
6. Train/test split
7. Clustering (K-Means + KNN)
8. Regression (Linear, Random Forest, SVR)
9. Evaluation & comparison
10. Multi-Layer Perceptron (MLP)

## 1. Setup

Import libraries, fix the random seed for reproducibility, and set consistent
plotting defaults.

In [2]:
# Stage 1: Setup - imports, reproducibility, plotting defaults


import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility: fix the random seed everywhere so results are
# identical on every rerun (required for a reproducible submission).
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot defaults — applied once so every figure looks consistent.
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["figure.dpi"] = 100

# Show all dataframe columns when printing (we have 10+).
pd.set_option("display.max_columns", None)

print("Setup complete.")
print("pandas:", pd.__version__, "| numpy:", np.__version__)

Setup complete.
pandas: 3.0.3 | numpy: 2.4.6


## 2. Data Loading

Load the California Housing dataset reproducibly: use the local copy in `data/`
if present, otherwise download it from the original source. This lets the
notebook run on a clean machine while keeping a version-controlled local copy.

In [3]:
# Stage 2: Data loading (reproducible: local copy or download)


import tarfile
import urllib.request

DATA_DIR = "../data"                       # repo's data/ folder
CSV_PATH = os.path.join(DATA_DIR, "housing.csv")
SOURCE_URL = "https://github.com/ageron/data/raw/main/housing.tgz"

def load_housing_data():
    """Return the housing DataFrame, downloading the dataset if needed."""
    if not os.path.exists(CSV_PATH):
        print("Local CSV not found — downloading from source...")
        os.makedirs(DATA_DIR, exist_ok=True)
        tgz_path = os.path.join(DATA_DIR, "housing.tgz")
        urllib.request.urlretrieve(SOURCE_URL, tgz_path)
        with tarfile.open(tgz_path) as tgz:
            tgz.extractall(path=DATA_DIR)
        # The archive extracts to data/housing/housing.csv — move it up.
        extracted = os.path.join(DATA_DIR, "housing", "housing.csv")
        if os.path.exists(extracted):
            os.replace(extracted, CSV_PATH)
        print("Download complete.")
    else:
        print("Loading local copy from", CSV_PATH)
    return pd.read_csv(CSV_PATH)

housing = load_housing_data()
print("Loaded:", housing.shape[0], "rows ×", housing.shape[1], "columns")
housing.head()

Loading local copy from ../data/housing.csv
Loaded: 20640 rows × 10 columns


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


## 3. Exploratory Data Analysis

Before any preprocessing, inspect the dataset to understand its structure and
identify issues that must be handled. We specifically check for:

- **Feature types** — nine numeric features and one categorical (`ocean_proximity`).
- **Missing values** — which columns, how many.
- **Distribution artefacts** — capped values that are census artefacts, not true data.
- **Categorical balance** — how the `ocean_proximity` levels are distributed.

These findings directly justify the preprocessing decisions made in Section 4.

In [4]:
# Stage 3a: Structure, dtypes, and summary statistics



print("Dataset shape:", housing.shape)
print("\n--- Column types ---")
print(housing.dtypes)

print("\n--- Summary statistics (numeric features) ---")
display(housing.describe())

Dataset shape: (20640, 10)

--- Column types ---
longitude             float64
latitude              float64
housing_median_age    float64
total_rooms           float64
total_bedrooms        float64
population            float64
households            float64
median_income         float64
median_house_value    float64
ocean_proximity           str
dtype: object

--- Summary statistics (numeric features) ---


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


In [5]:
# Stage 3b: Missing values



missing = housing.isnull().sum()
missing_pct = (missing / len(housing) * 100).round(2)
missing_table = pd.DataFrame({"missing_count": missing, "missing_%": missing_pct})
print("Missing values per column:")
display(missing_table[missing_table["missing_count"] > 0])

print(f"\nTotal rows: {len(housing)}")
print("Only 'total_bedrooms' has missing values "
      f"({missing['total_bedrooms']} rows, "
      f"{missing_pct['total_bedrooms']}% of data).")

Missing values per column:


,missing_count,missing_%
total_bedrooms,207,1.0



Total rows: 20640
Only 'total_bedrooms' has missing values (207 rows, 1.0% of data).
